# MOUSE CORE · BGE-M3 pilot

Pilotní výpočet skutečných textových embeddingů pro tři původní karty z repozitáře `generative_gen`: Myš, Ryba a Kocour.

**Co notebook dělá:** stáhne původní české texty z GitHubu, zachová jejich zdrojové soubory, rozdělí je na přiměřeně velké textové úseky, načte model `BAAI/bge-m3` a vypočítá dense vektory. Výsledný JSON lze importovat do MOUSE CORE → Vector Lab.

**Důležité:** BGE-M3 je zde použit pouze pro text. Obrázky jsou evidovány jako součást karet a jejich odkazy zůstávají ve výstupu, ale tento model z nich obrazové vektory nevytváří. Obrazová analýza bude samostatný krok.

V Kaggle zapni Internet a zvol GPU akcelerátor. Pak spusť notebook přes **Run All**.

In [ ]:
!pip -q install -U FlagEmbedding

In [ ]:
import json, re, csv, os, sys, subprocess
from pathlib import Path
from urllib.request import urlopen
import torch

REPO = 'https://raw.githubusercontent.com/artmisanovak-prog/generative_gen/main'
MODEL_NAME = 'BAAI/bge-m3'
CARDS = {
    'mys': {'name': 'Myš', 'text': 'doma/primarni_data/kriva_pohadka/mys.txt', 'image': 'doma/primarni_data/ilustrace/mys.jpg'},
    'ryba': {'name': 'Ryba', 'text': 'doma/primarni_data/kriva_pohadka/ryba.txt', 'image': 'doma/primarni_data/ilustrace/ryba.jpg'},
    'kocour': {'name': 'Kocour', 'text': 'doma/primarni_data/kriva_pohadka/kocour.txt', 'image': 'doma/primarni_data/ilustrace/kocour.jpg'},
}
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print('Zařízení:', 'GPU' if torch.cuda.is_available() else 'CPU')
print('Výstupní složka:', OUT)

## 1. Načtení původních zdrojů
Texty se stahují přímo z hlavní větve repozitáře. Notebook je neupravuje ani nepřekládá.

In [ ]:
documents = []
for card_id, info in CARDS.items():
    url = f"{REPO}/{info['text']}"
    raw = urlopen(url).read().decode('utf-8-sig')
    image_url = f"{REPO}/{info['image']}"
    documents.append({
        'card_id': card_id,
        'card_name': info['name'],
        'source_file': info['text'],
        'illustration_file': info['image'],
        'illustration_url': image_url,
        'original_text': raw.strip()
    })
    print(f"Načteno: {info['name']} | {len(raw)} znaků | {info['text']}")

assert len(documents) == 3
print('\nVšechny tři původní texty jsou načtené.')

## 2. Segmentace textů
Vedle embeddingu celého textu vzniknou také embeddingy větších navazujících bloků. Zachováváme pořadí a původní české znění. Drobné odstavce se slučují do bloků přibližně do 1 200 znaků, aby se neztratila návaznost.

In [ ]:
def make_chunks(text, target_chars=1200):
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n+', text) if p.strip()]
    chunks, current = [], ''
    for paragraph in paragraphs:
        candidate = paragraph if not current else current + '\n\n' + paragraph
        if current and len(candidate) > target_chars:
            chunks.append(current)
            current = paragraph
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks

items = []
for doc in documents:
    items.append({
        'id': f"{doc['card_id']}-full",
        'card_id': doc['card_id'], 'card_name': doc['card_name'],
        'kind': 'full_text', 'text': doc['original_text'],
        'source_file': doc['source_file'],
        'illustration_file': doc['illustration_file'],
        'illustration_url': doc['illustration_url']
    })
    for i, chunk in enumerate(make_chunks(doc['original_text']), start=1):
        items.append({
            'id': f"{doc['card_id']}-segment-{i:02d}",
            'card_id': doc['card_id'], 'card_name': doc['card_name'],
            'kind': 'text_segment', 'segment_index': i, 'text': chunk,
            'source_file': doc['source_file'],
            'illustration_file': doc['illustration_file'],
            'illustration_url': doc['illustration_url']
        })

print('Počet položek k embedování:', len(items))
for d in documents:
    print(d['card_name'], '→', sum(1 for x in items if x['card_id']==d['card_id']), 'položek včetně celého textu')

## 3. Načtení BGE-M3 a výpočet vektorů
Model se stáhne do dočasného prostředí Kaggle. Nemusíš ho instalovat do telefonu ani počítače. Pokud GPU není dostupné, notebook poběží na CPU pomaleji.

In [ ]:
from FlagEmbedding import BGEM3FlagModel

use_fp16 = torch.cuda.is_available()
model = BGEM3FlagModel(MODEL_NAME, use_fp16=use_fp16)
texts = [item['text'] for item in items]
encoded = model.encode(
    texts,
    batch_size=4,
    max_length=8192,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)
vectors = encoded['dense_vecs']
print('Vypočteno vektorů:', len(vectors))
print('Dimenze jednoho vektoru:', len(vectors[0]))

## 4. Export pro MOUSE CORE
Výstupní soubor `mouse_core_bge_m3_vectors.json` je přímo určený pro import do Vector Lab. Každá položka obsahuje text, vektor, typ položky, původní soubor a odkaz na ilustraci.

In [ ]:
export_rows = []
for item, vector in zip(items, vectors):
    row = dict(item)
    row.update({'model': MODEL_NAME, 'vector': [float(x) for x in vector]})
    export_rows.append(row)

json_path = OUT / 'mouse_core_bge_m3_vectors.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(export_rows, f, ensure_ascii=False)

manifest = {
    'experiment': 'MOUSE CORE BGE-M3 pilot',
    'model': MODEL_NAME,
    'model_type': 'text_embedding',
    'language': 'Czech',
    'cards': [d['card_id'] for d in documents],
    'item_count': len(export_rows),
    'vector_dimension': len(vectors[0]),
    'source_repository': 'https://github.com/artmisanovak-prog/generative_gen',
    'note': 'Vectors represent text only. Illustration URLs are provenance references, not image embeddings.'
}
manifest_path = OUT / 'mouse_core_bge_m3_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print('Hotovo:')
print(json_path)
print(manifest_path)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 5. Kontrola výsledku
Zkontrolujeme počet vektorů, dimenzi a přítomnost všech tří karet. Samotná podobnost vektorů není důkazem porozumění ani přenosu pravidla; je to pouze jedna měřitelná vrstva experimentu.

In [ ]:
from collections import Counter
print('Počty podle karet:', dict(Counter(x['card_name'] for x in export_rows)))
print('Počty podle typu:', dict(Counter(x['kind'] for x in export_rows)))
assert all(len(x['vector']) == len(vectors[0]) for x in export_rows)
assert {x['card_id'] for x in export_rows} == {'mys', 'ryba', 'kocour'}
print('Kontrola prošla.')

# V Kaggle otevři panel Output a stáhni mouse_core_bge_m3_vectors.json.